In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 02 - Bronze Ingestion
# MAGIC
# MAGIC This notebook ingests raw synthetic healthcare CSV files from Unity Catalog Volumes into Bronze Delta tables.
# MAGIC
# MAGIC The Bronze layer preserves raw data with minimal transformation while adding ingestion metadata.
# MAGIC
# MAGIC ## Bronze Tables Created
# MAGIC - bronze_members
# MAGIC - bronze_providers
# MAGIC - bronze_labs
# MAGIC - bronze_medications
# MAGIC - bronze_claims

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col
from pyspark.sql.utils import AnalysisException

In [0]:
# Unity Catalog settings
CATALOG_NAME = "workspace"
SCHEMA_NAME = "default"
VOLUME_NAME = "healthcare_data"

# Base path for project files
VOLUME_BASE_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}"

# Raw data path where Notebook 01 saved the CSV files
RAW_DATA_PATH = f"{VOLUME_BASE_PATH}/raw"

print(f"Volume base path: {VOLUME_BASE_PATH}")
print(f"Raw data path: {RAW_DATA_PATH}")

In [0]:
# Target catalog and schema for Delta tables
TARGET_CATALOG = CATALOG_NAME
TARGET_SCHEMA = SCHEMA_NAME

print(f"Target catalog: {TARGET_CATALOG}")
print(f"Target schema: {TARGET_SCHEMA}")

In [0]:
display(dbutils.fs.ls(RAW_DATA_PATH))

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_CATALOG}.{TARGET_SCHEMA}")

spark.sql(f"USE CATALOG {TARGET_CATALOG}")
spark.sql(f"USE SCHEMA {TARGET_SCHEMA}")

print(f"Using schema: {TARGET_CATALOG}.{TARGET_SCHEMA}")

In [0]:
raw_files = {
    "members": {
        "path": f"{RAW_DATA_PATH}/members.csv",
        "table_name": f"{TARGET_CATALOG}.{TARGET_SCHEMA}.bronze_members"
    },
    "providers": {
        "path": f"{RAW_DATA_PATH}/providers.csv",
        "table_name": f"{TARGET_CATALOG}.{TARGET_SCHEMA}.bronze_providers"
    },
    "labs": {
        "path": f"{RAW_DATA_PATH}/labs.csv",
        "table_name": f"{TARGET_CATALOG}.{TARGET_SCHEMA}.bronze_labs"
    },
    "medications": {
        "path": f"{RAW_DATA_PATH}/medications.csv",
        "table_name": f"{TARGET_CATALOG}.{TARGET_SCHEMA}.bronze_medications"
    },
    "claims": {
        "path": f"{RAW_DATA_PATH}/claims.csv",
        "table_name": f"{TARGET_CATALOG}.{TARGET_SCHEMA}.bronze_claims"
    }
}

raw_files

In [0]:
def ingest_csv_to_bronze(source_name, source_path, target_table):
    """
    Reads a raw CSV file from Unity Catalog Volumes and writes it to a Bronze Delta table.

    Parameters:
    source_name: Name of the source dataset
    source_path: Path to the raw CSV file
    target_table: Fully qualified Delta table name
    """

    print(f"Starting Bronze ingestion for: {source_name}")
    print(f"Source path: {source_path}")
    print(f"Target table: {target_table}")

    try:
        raw_df = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .option("multiLine", True)
            .option("escape", '"')
            .csv(source_path)
        )

        bronze_df = (
            raw_df
            .withColumn("source_system", lit("synthetic_healthcare_data"))
            .withColumn("source_dataset", lit(source_name))
            .withColumn("source_file", col("_metadata.file_path"))
            .withColumn("ingestion_timestamp", current_timestamp())
        )

        (
            bronze_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", True)
            .saveAsTable(target_table)
        )

        record_count = bronze_df.count()

        print(f"Successfully created {target_table}")
        print(f"Record count: {record_count}")
        print("-" * 80)

        return {
            "source_name": source_name,
            "target_table": target_table,
            "record_count": record_count,
            "status": "Success"
        }

    except Exception as error:
        print(f"Failed to ingest {source_name}")
        print(f"Error: {error}")
        print("-" * 80)

        return {
            "source_name": source_name,
            "target_table": target_table,
            "record_count": 0,
            "status": "Failed",
            "error": str(error)
        }

In [0]:
ingestion_results = []

for source_name, config in raw_files.items():
    result = ingest_csv_to_bronze(
        source_name=source_name,
        source_path=config["path"],
        target_table=config["table_name"]
    )

    ingestion_results.append(result)

In [0]:
results_df = spark.createDataFrame(ingestion_results)

display(results_df)

In [0]:
spark.sql(f"SHOW TABLES IN {TARGET_CATALOG}.{TARGET_SCHEMA} LIKE 'bronze_*'").show(truncate=False)

In [0]:
display(spark.table(f"{TARGET_CATALOG}.{TARGET_SCHEMA}.bronze_members").limit(10))

In [0]:
display(spark.table(f"{TARGET_CATALOG}.{TARGET_SCHEMA}.bronze_labs").limit(10))

In [0]:
display(spark.table(f"{TARGET_CATALOG}.{TARGET_SCHEMA}.bronze_claims").limit(10))

In [0]:
bronze_tables = [
    "bronze_members",
    "bronze_providers",
    "bronze_labs",
    "bronze_medications",
    "bronze_claims"
]

for table in bronze_tables:
    full_table_name = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{table}"
    count = spark.table(full_table_name).count()
    print(f"{full_table_name}: {count:,} records")

In [0]:
for table in bronze_tables:
    full_table_name = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{table}"

    print(f"\nSchema for {full_table_name}")
    print("-" * 80)

    spark.table(full_table_name).printSchema()

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

def show_null_counts(table_name):
    df = spark.table(table_name)

    null_counts = df.select([
        spark_sum(col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ])

    print(f"Null counts for {table_name}")
    display(null_counts)


show_null_counts(f"{TARGET_CATALOG}.{TARGET_SCHEMA}.bronze_members")
show_null_counts(f"{TARGET_CATALOG}.{TARGET_SCHEMA}.bronze_labs")
show_null_counts(f"{TARGET_CATALOG}.{TARGET_SCHEMA}.bronze_claims")

In [0]:
# MAGIC %sql
# MAGIC SELECT *
# MAGIC FROM workspace.default.bronze_members
# MAGIC LIMIT 10;